In [1]:
pwd

'C:\\Users\\hamza\\Programs\\traffic-sensor-ai\\notebooks'

In [2]:
cd ..

C:\Users\hamza\Programs\traffic-sensor-ai


C:\Users\hamza\anaconda3\envs\smart_sensor\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import paho.mqtt.client as mqtt
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)

In [5]:
client.connect("localhost", 1883)

<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

In [6]:
from communication.mqtt_client import MqttClientConfig, SmartSensorMqttClient


In [7]:
from communication.services import ConfigurationPublisherService, ConfigurationPublisherService, ConfigurationReceiverService, SnapshotCommandService, SnapshotReceiverService
from communication.topics import SensorTopics


topics = SensorTopics(sensor_id="camera_1")



sensor_config = MqttClientConfig(
    broker_host="localhost",
    client_id="camera_1",
    offline_queue_dir="outputs/pending_mqtt/camera_1",

)

platform_config = MqttClientConfig(
    broker_host="localhost",
    client_id="dashboard",
    offline_queue_dir="outputs/pending_mqtt/dashboard",

)








def handle_sensor_message(topic, payload):
    print(f"[Sensor] Received topic: {topic}")

    if topic == topics.configuration_command:
        saved_path = config_receiver_service.save_configuration_file(payload)
        print("Configuration saved to:", saved_path)

    if topic == topics.configuration_command:
        print("[Sensor] Handling configuration request...")
        saved_path = config_receiver_service.save_configuration_file(payload)
        platform_client.publish_json(
            topic=topics.configuration_response,
            payload={"status": "success", "saved_path": saved_path},
        )

    
    elif topic == topics.snapshot_command:
        print("[Sensor] Handling snapshot request...")
        snapshot_service.handle_snapshot_request(payload)



def handle_platform_message(topic, payload):
    if topic == topics.snapshot_response:
        saved_path = snapshot_receiver_service.save_snapshot(payload)
        print("Snapshot saved to:", saved_path)

    elif topic == topics.configuration_response:
        print("[Platform] Configuration acknowledged.")




sensor_client = SmartSensorMqttClient(
    sensor_config,
    on_message=handle_sensor_message,
)


platform_client = SmartSensorMqttClient(
    platform_config,
    on_message=handle_platform_message,
)


config_publish_service = ConfigurationPublisherService(platform_client, topics)
config_receiver_service = ConfigurationReceiverService()
snapshot_service = SnapshotCommandService(platform_client, topics)
snapshot_receiver_service = SnapshotReceiverService()



In [8]:
sensor_client.connect()
platform_client.connect()

[Sensor] Received topic: sensors/camera_1/commands/configuration
Configuration saved to: C:\Users\hamza\Programs\traffic-sensor-ai\config\traffic_metrics.yaml
[Sensor] Handling configuration request...
[Platform] Configuration acknowledged.
[Sensor] Received topic: sensors/camera_1/commands/snapshot
[Sensor] Handling snapshot request...
[Snapshot] Request received
Snapshot saved to: C:\Users\hamza\Programs\traffic-sensor-ai\outputs\snapshots\snapshot_20260701_190732.jpg


In [9]:
sensor_client.subscribe(topics.configuration_command)
sensor_client.subscribe(topics.snapshot_command)

platform_client.subscribe(topics.configuration_response)
platform_client.subscribe(topics.snapshot_response)

In [10]:
print(platform_client.connected)
print(sensor_client.connected)

True
True


In [11]:
config_publish_service.publish_configuration_file()

True

In [12]:
platform_client.publish_json(
    topic=topics.snapshot_command,
    payload={"request": "snapshot"},
)

True

In [ ]:
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"Nom du GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
source = r"C:\Users\hamza\Datasets\TrafficDatasets\IMAROC_2\kech38.mp4"

In [ ]:
# from utils.shape_setter import PointSelector , LineSelector , TwoLineSelector , PolygonSelector , RectangleSelector , OBBSelector
# import cv2
# import numpy as np

# stride = 1
# stride_method = "periodic_stride"             # "burst_stride", "periodic_stride", "random_sampling"

# cap = cv2.VideoCapture(source)
# ok, first_frame = cap.read()
# cap.release()

# if not ok or first_frame is None:
#     print("Failed to read example frame; please provide a valid path ('kech.mp4' used in example).")
# else:
#     # line
#     line_sel = LineSelector(max_display_size=900, auto_confirm=True, preview_wait_secs=None)
#     line = line_sel.select_line(first_frame)
#     print("Selected line:", line)

#     # # two lines
#     # two_sel = TwoLineSelector(max_display_size=900, auto_confirm=True, preview_wait_secs=None)
#     # two = two_sel.select_two_lines(first_frame)
#     # print("Selected two lines:", two)

#     # #polygon
#     # poly_sel = PolygonSelector(max_display_size=900, min_points=4, auto_close_on_click_near_first=True,
#     #                            close_pixel_radius=12, preview_wait_secs=None)
#     # poly = poly_sel.select_polygon(first_frame)
#     # print("Selected polygon:", poly)

#     # # rectangle
#     # rect_sel = RectangleSelector(max_display_size=900, auto_confirm=True, preview_wait_secs=None)
#     # rect = rect_sel.select_rectangle(first_frame)
#     # if rect is None:
#     #     print("Cancelled")
#     # else:
#     #     print("Selected rectangle:", rect)


#     # # # obb£
#     # obb_selector = OBBSelector()
#     # obb_points = obb_selector.select_obb(first_frame)
#     # if obb_points is not None:
#     #     print("Selected OBB points:", obb_points)
#     # else:
#     #     print("Selection cancelled")


#     # selector = PointSelector()
#     # pt = selector.select_point(first_frame)
#     # print("Selected point:", pt)


In [ ]:
from config.load_build import load_config, build_areas, build_runtime_components

In [ ]:
config_path = "config/traffic_metrics.yaml"
cfg = load_config(config_path)

In [ ]:
areas = build_areas(cfg)

In [ ]:
geometry_engine , crossing_estimator = build_runtime_components(areas, cfg)

In [ ]:
from detection.factory import build_detector

In [ ]:
detector = build_detector(
    model_name=cfg["detector"]["model_name"],
    conf=cfg["detector"]["confidence"],
)

In [ ]:
detector.predictor.args.conf , detector.predictor.args.imgsz

In [ ]:
from tracking.track import Tracker

tracker = Tracker(
    method=cfg["tracker"]["method"],
    reid_model=cfg["tracker"]["reid_model"],
    classes=cfg["tracker"]["classes"],
    device=cfg["tracker"]["device"],
    half=cfg["tracker"]["half"],
    per_class=cfg["tracker"]["per_class"],
)

In [ ]:
from utils.profilers import Profile
device = detector.predictor.device
traffic_metrics_profile = Profile()
det_profile = Profile(device=device)
track_profile = Profile(device=device)

In [ ]:
import numpy as np

def extract_tracking_outputs(
    tracks: np.ndarray,
):

    bboxes = tracks[:, :4]

    points = np.empty((bboxes.shape[0], 2), dtype=np.float32)

    points[:, 0] = (bboxes[:, 0] + bboxes[:, 2]) * 0.5
    points[:, 1] = bboxes[:, 3]

    track_ids = tracks[:, 4].astype(np.int32)

    det_cls = tracks[:, 6].astype(np.int32)

    # print(track_ids.shape)
    # print(det_cls.shape)
    
    return points, bboxes, track_ids, det_cls

In [ ]:
def compute_area_metrics(
    areas,
    points,
    track_ids,
    det_cls,
    current_time,
    line_cache,
    polygon_cache,
    crossed_masks,
    new_crossings,
):

    results = {}

    for area in areas:

        area_results = {}

        # ---------------------------------
        # Polygon cache
        # ---------------------------------

        polygon_mask = None

        if area.zone is not None:
            polygon_mask = polygon_cache[area.zone.polygon_id]

        # ---------------------------------
        # Line cache
        # ---------------------------------

        crossed_mask = None
        vicinity_mask = None
        new_area_crossings = []

        if area.flow_line is not None:

            line_idx = geometry_engine._line_id_to_idx[
                area.flow_line.line_id
            ]
            crossed_mask = crossed_masks[line_idx]
            # print(f"line_idx:{line_idx}")
            vicinity_mask = line_cache["vicinity_mask"][line_idx]
            # print(f"vicinity_mask.shape after line_cache['vicinity_mask'][line_idx] is {vicinity_mask.shape}")
            new_area_crossings = new_crossings.get(
                area.area_id,
                [],
            )

        # ---------------------------------
        # Flow (+ internal counter)
        # ---------------------------------

        if "flow" in area.metrics:

            counter_res = area.metrics["counter"].compute(
                track_ids=track_ids,
                det_cls=det_cls,
                current_time=current_time,
                crossed_mask=crossed_mask,
                vicinity_mask=vicinity_mask,
                polygon_mask=polygon_mask,
            )

            area_results["flow"] = (
                area.metrics["flow"].compute(
                    cumulative_count=counter_res.cumulative_count,
                    cumulative_counts_by_class=counter_res.cumulative_counts_by_class,
                    current_time=current_time,
                )
            )

        # ---------------------------------
        # Density
        # ---------------------------------

        if "density" in area.metrics:

            area_results["density"] = (
                area.metrics["density"].compute(
                    polygon_mask=polygon_mask,
                    det_cls=det_cls,
                )
            )

        # ---------------------------------
        # Occupancy
        # ---------------------------------

        if "occupancy" in area.metrics:

            area_results["occupancy"] = (
                area.metrics["occupancy"].compute(
                    vicinity_mask=vicinity_mask,
                    polygon_mask=polygon_mask,
                )
            )

        # ---------------------------------
        # Space headway
        # ---------------------------------

        if "space_headway" in area.metrics:

            area_results["space_headway"] = (
                area.metrics["space_headway"].compute(
                    points=points,
                    polygon_mask=polygon_mask,
                )
            )

        # ---------------------------------
        # Time headway
        # ---------------------------------

        if "time_headway" in area.metrics:

            area_results["time_headway"] = (
                area.metrics["time_headway"].compute(
                    new_area_crossings
                )
            )

        results[area.area_id] = area_results

    return results

In [ ]:
from video_io.frame_producer import FrameProducer, RealTimeSimulationProducer, OfflineSampledFrameProducer, DirectFrameProducer


producer = DirectFrameProducer(source)
producer.start()

In [ ]:
fps = 30
try:
    with torch.inference_mode():
        while True:
            # try to get a frame but don't block forever
            frame = producer.next_frame() 
            if frame is None:
                break
            else:
                print(f"frame_grabber index: {frame.read_idx}")

                current_time = frame.read_idx / fps

                with det_profile: 
                    ready_to_track_array = detector.detect_to_track(frame.data)
                with track_profile: 
                    res = tracker.update(ready_to_track_array , frame.data)
    
                with traffic_metrics_profile:
                    points, bboxes, track_ids, det_cls = extract_tracking_outputs(res)
                    line_cache, polygon_cache = geometry_engine.compute(points, bboxes)
                    crossed_masks, new_crossings = crossing_estimator.update(track_ids, current_time, line_cache, polygon_cache)
    
    
                    area_metrics = compute_area_metrics(
                                                        areas=areas,
                                                        points=points,
                                                        track_ids=track_ids,
                                                        det_cls=det_cls,
                                                        current_time=current_time,
                                                        line_cache=line_cache,
                                                        polygon_cache=polygon_cache,
                                                        crossed_masks=crossed_masks,
                                                        new_crossings=new_crossings,
                                                    )
            

                
            # break

except KeyboardInterrupt:
    pass

In [ ]:
area_metrics

In [ ]:
areas

In [ ]:
areas[0].metrics["counter"].cumulative_count , areas[0].metrics["counter"].cumulative_counts_by_class

In [ ]:
area_metrics["area_1"]["density"]

In [ ]:
det_profile.dt*1000  , track_profile.dt*1000  , traffic_metrics_profile.dt*1000 

In [ ]:
det_profile.t  , track_profile.t  , traffic_metrics_profile.t 

In [ ]:
(traffic_metrics_profile.t/(det_profile.t + track_profile.t + traffic_metrics_profile.t)) * 100

In [ ]:
(traffic_metrics_profile.dt/(det_profile.dt + track_profile.dt + traffic_metrics_profile.dt)) * 100

In [ ]:
ready_to_track_array

In [ ]:
res

In [ ]:
line_cache, polygon_cache

In [ ]:
crossed_masks, new_crossings

In [ ]:
areas[0].metrics

In [ ]:
areas[0]